# 预处理

In [23]:
import os
from scapy.utils import rdpcap
from scapy.layers.inet import IP, TCP
from scapy.layers.tls import *
import torch  # For tensor storage

In [ ]:
# Helper function to extract flow identifier
def get_flow_id(packet):
    ip_layer = packet[IP]
    tcp_layer = packet[TCP]
    return (ip_layer.src, tcp_layer.sport, ip_layer.dst, tcp_layer.dport)

def get_bidirectional_flow_id(packet):
    ip_layer = packet[IP]
    tcp_layer = packet[TCP]
    # Create a flow identifier
    flow_id = (ip_layer.src, tcp_layer.sport, ip_layer.dst, tcp_layer.dport)
    reverse_flow_id = (ip_layer.dst, tcp_layer.dport, ip_layer.src, tcp_layer.sport)
    # Return the lexicographically smaller tuple to ensure consistency
    return min(flow_id, reverse_flow_id)

# Helper function to extract packet length
def get_packet_length(packet):
    return len(packet)

# Helper function to extract packet timestamp
def get_packet_timestamp(packet):
    return packet.time

# Helper function to extract TCP flags
def get_tcp_flags(packet):
    return packet[TCP].flags

# Helper function to extract Payload Len
def get_payload_length(packet):
    if TCP in packet:
        # Get the payload of the TCP layer
        payload = packet[TCP].payload
        if payload:
            print("Payload exists:", bytes(payload))
        else:
            print("No payload in this packet.")
        # Return the length of the payload
        return len(payload)
    return 0  # Return 0 if no payload exists

# Helper function to determine if a packet is uplink or downlink
def is_uplink(packet, flow_id):
    """
    Determine if a packet is uplink (client to server) or downlink (server to client).

    Args:
        packet: A Scapy packet object.
        flow_id: A tuple (src_ip, src_port, dst_ip, dst_port) representing the flow.

    Returns:
        str: "uplink" if the packet is client to server, "downlink" if server to client.
    """
    src_ip, src_port, dst_ip, dst_port = flow_id

    # Check if the packet matches the uplink direction
    if packet[IP].src == src_ip and packet[TCP].sport == src_port:
        return "uplink"  # Client to server

    # Check if the packet matches the downlink direction
    if packet[IP].src == dst_ip and packet[TCP].sport == dst_port:
        return "downlink"  # Server to client

    return "unknown"  # If it doesn't match either direction

# 提取流的 握手包内容, 需要具体内容
def extract_handshake_payload(packet):
    # Read packets from the pcap file
    handshake_packets = []
    # Check if the packet has IP and TCP layers
    if IP in packet and TCP in packet:
        tcp_layer = packet[TCP]
        # Check for SYN and ACK flags
        if tcp_layer.flags & 0x02:
            handshake_packets.append(packet)
        elif tcp_layer.flags & 0x10:
            handshake_packets.append(packet)
    # Return the handshake packets
    return handshake_packets

# 提取流信息的函数
def extract_flows(pcap_file, extract_features=None):
    """
    Extract packet length sequences for each TCP flow from a pcap file.

    Args:
        pcap_file (str): Path to the pcap file.
        extract_features (list): List of features to extract from packets.

    Returns:
        dict: A dictionary where keys are flow identifiers (e.g., tuple of IPs and ports)
              and values are lists of packet lengths.
    """
    if not os.path.exists(pcap_file):
        raise FileNotFoundError(f"PCAP file not found: {pcap_file}")

    # Read packets from the pcap file
    packets = rdpcap(pcap_file)

    flows = {}  # Dictionary to store flows and their packet lengths
    # 切分成流
    for packet in packets:
        # Check if the packet has IP and TCP layers
        if IP in packet and TCP in packet:
            flow_id = get_bidirectional_flow_id(packet)
            if flow_id not in flows:
                flows[flow_id] = []
            # Extract requested features
            packet_info = {}
            if "length" in extract_features:
                packet_info["length"] = get_packet_length(packet)
            if "timestamp" in extract_features:
                packet_info["timestamp"] = get_packet_timestamp(packet)
            if "flags" in extract_features:
                packet_info["flags"] = get_tcp_flags(packet)
            if "handshake" in extract_features:
                handshake_payload = extract_handshake_payload(packet)
                packet_info["handshake_payload"] = handshake_payload
            if "payload_length" in extract_features:
                packet_info["payload_length"] = get_payload_length(packet)
            if "direction" in extract_features:
                packet_info["direction"] = is_uplink(packet, flow_id)
            # Append the packet info to the corresponding flow
            flows[flow_id].append(packet_info)

    return flows

In [25]:
# 保存成 tensor 张量
def save_flows_as_tensors(flows, output_dir):
    """
    Save flows as tensors to the specified directory.

    Args:
        flows (dict): A dictionary where keys are flow identifiers and values are lists of packet info dicts.
        output_dir (str): Directory to save the tensors.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for i, (flow_id, packets) in enumerate(flows.items()):
        # Convert packet info to a PyTorch tensor
        tensor_data = {
            'length': [packet.get('length', 0) for packet in packets],
            'timestamp': [packet.get('timestamp', 0) for packet in packets],
            'flags': [packet.get('flags', 0) for packet in packets],
            'handshake_payload': [len(packet.get('handshake_payload', [])) for packet in packets],
            'payload_length': [packet.get('payload_length', 0) for packet in packets],
            'direction': [1 if packet.get('direction') == 'uplink' else 0 for packet in packets]
        }
        tensor = {key: torch.tensor(value, dtype=torch.float32) for key, value in tensor_data.items()}

        # Save the tensor to a file
        flow_file = os.path.join(output_dir, f"flow_{i}.pt")
        torch.save(tensor, flow_file)

        print(f"Saved flow {flow_id} to {flow_file}")

In [ ]:
if __name__ == "__main__":
    # Example usage
    pcap_file = "./handshark.pcap"  # Replace with your pcap file path
    output_dir = "output_tensors"  # Directory to save tensors

    # Extract packet lengths for each TCP flow
    flows = extract_flows(pcap_file, extract_features=["length", "payload_length"])

    # Save flows as tensors
    save_flows_as_tensors(flows, output_dir)

Saved flow ('10.161.34.27', 22, '10.21.202.2', 2294) to output_tensors/flow_0.pt
Saved flow ('10.21.202.2', 2730, '183.240.228.248', 14021) to output_tensors/flow_1.pt
Saved flow ('10.21.202.2', 3003, '111.124.200.68', 443) to output_tensors/flow_2.pt
Saved flow ('10.21.202.2', 2329, '116.62.121.9', 1119) to output_tensors/flow_3.pt
Saved flow ('10.21.202.2', 2550, '183.240.228.248', 14011) to output_tensors/flow_4.pt
Saved flow ('10.21.202.2', 2737, '183.240.228.248', 14021) to output_tensors/flow_5.pt
Saved flow ('10.21.202.2', 2590, '120.232.91.90', 21510) to output_tensors/flow_6.pt
Saved flow ('10.21.202.2', 2853, '183.240.228.248', 14021) to output_tensors/flow_7.pt
Saved flow ('10.21.202.2', 3024, '10.3.9.4', 53) to output_tensors/flow_8.pt
Saved flow ('10.21.202.2', 3025, '10.3.9.4', 53) to output_tensors/flow_9.pt
Saved flow ('10.21.202.2', 3026, '10.3.9.4', 53) to output_tensors/flow_10.pt
Saved flow ('10.21.202.2', 3027, '159.89.41.53', 443) to output_tensors/flow_11.pt
Save

In [27]:
# 加载张量
tensor = torch.load("output_tensors/flow_1.pt")
print(tensor)  # 输出张量内容

{'length': tensor([55., 66., 55., 66.]), 'timestamp': tensor([0., 0., 0., 0.]), 'flags': tensor([0., 0., 0., 0.]), 'handshake_payload': tensor([0., 0., 0., 0.]), 'payload_length': tensor([0., 0., 0., 0.]), 'direction': tensor([0., 0., 0., 0.])}
